In [5]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib import rc

## Question 1

### Question 1a

In [91]:
# import polblog data
G = nx.DiGraph()
with open('data/polblogs_nodes_class.tsv','r') as fp:
    for line in fp:
        node_id, node_label, node_political = line.strip().split('\t')
        political_orientation = 'liberal' if node_political == '0' else 'conservative'
        G.add_node(node_id, website=node_label, political_orientation=political_orientation)

with open('data/polblogs_edges_class.tsv','r') as fp:
    for line in fp:
        source_node_id, target_node_id = line.strip().split('\t')
        G.add_edge(source_node_id, target_node_id)

In [92]:
print("Number of nodes: %d" % G.number_of_nodes())
print("Number of edges: %d" % G.number_of_edges())
print("Graph density:\t %1.7f" % nx.density(G))

Number of nodes: 1490
Number of edges: 19025
Graph density:	 0.0085752


In [93]:

# Write all the edgelists
import json
import pickle

# Edgelist
nx.write_edgelist(G, "outputs/polblogs.edgelist")

# GraphML
nx.write_graphml(G, "outputs/polblogs.graphml")

#JSON
json_graph = nx.node_link_data(G)
with open("outputs/polblogs.json", "w", encoding="utf-8") as gfile:
    json.dump(json_graph, gfile)

# Pickle binary serialization
with open("outputs/polblogs.pkl", 'wb') as gfile:
    pickle.dump(G, gfile)

# dense adjacency matrix
nmp = nx.to_numpy_array(G)
with open("outputs/polblogs.npy", 'wb') as gfile:
    np.save(gfile, nmp)


In [ ]:
# Read in data
import os

# First, write the original network's information
# Now save everything to a file for easy viewing 
with open("outputs/network_info.txt", 'w') as info_file:
    info_file.write(f"Network: G \n")
    info_file.write(f"Node Count: {G.number_of_nodes()}\n")
    info_file.write(f"Edge Count: {G.number_of_edges()}\n")
    info_file.write(f"Is directed: {nx.is_directed(G)}\n")
    info_file.write(f"Website Attr Length: {len(nx.get_node_attributes(G, 'website'))}\n")
    info_file.write(f"Political Orientation Attr Length: {len(nx.get_node_attributes(G, 'political_orientation'))}\n")
    info_file.write(f"Graphs are equal: {nx.utils.graphs_equal(G, G)}\n") 
    info_file.write(f"\n\n")

network_list = list()
suffixes = ["edgelist", "graphml", "json", "pkl", "npy"]

# Edgelist
network_list.append(nx.read_edgelist("outputs/polblogs.edgelist"))

# GraphML
network_list.append(nx.read_graphml("outputs/polblogs.graphml"))

# JSON
with open('outputs/polblogs.json', 'r', encoding='utf-8') as file:
    node_link_data = json.load(file)
network_list.append(nx.node_link_graph(node_link_data))

# Pickle binary serialization
with open("outputs/polblogs.pkl", 'rb') as gfile:
    network_list.append(pickle.load(gfile))

# NP dense ajacency matrix
with open("outputs/polblogs.npy", 'rb') as gfile:
    back_np = np.load(gfile)
    network_list.append(nx.from_numpy_array(back_np))

# Now save everything to the file for easy viewing 
with open("outputs/network_info.txt", 'a') as info_file:
    for i, network in enumerate(network_list):
        info_file.write(f"Network: {suffixes[i]} \n")
        info_file.write(f"File size: {round(os.path.getsize(f'outputs/polblogs.{suffixes[i]}') / 1024)} kilobytes \n")
        info_file.write(f"Node Count: {network.number_of_nodes()}\n")
        info_file.write(f"Edge Count: {network.number_of_edges()}\n")
        info_file.write(f"Is directed: {nx.is_directed(network)}\n")
        info_file.write(f"Website Attr Length: {len(nx.get_node_attributes(network, 'website'))}\n")
        info_file.write(f"Political Orientation Attr Length: {len(nx.get_node_attributes(network, 'political_orientation'))}\n")
        info_file.write(f"Graphs are equal: {nx.utils.graphs_equal(G, network)}\n")
        info_file.write("\n\n")

### Question 1b

In [40]:
# Now check the specifics of the attributes 

# First, what is G?
print(f"Looking at network G")
attrs = nx.get_node_attributes(G, 'website')
print(f'Attribute storage type is {type(attrs)}')
print(f'Attribute type is {type(attrs["0"])}')
print(f"G nodes: {type(G.nodes)}")
print(("\n\n"))

# Now compare the others
for i, network in enumerate(network_list):
    attrs = nx.get_node_attributes(network, 'website')

    if(len(attrs) > 0):
        print(f"Looking at network {suffixes[i]}")
        print(f'Attribute storage type is {type(attrs)}')
        print(f'Attribute type is {type(attrs["0"])}')
        print(("\n"))

# Check np file
nparray = np.load('outputs/polblogs.npy')
nparray

Looking at network G
Attribute storage type is <class 'dict'>
Attribute type is <class 'str'>
G nodes: <class 'networkx.classes.reportviews.NodeView'>



Looking at network graphml
Attribute storage type is <class 'dict'>
Attribute type is <class 'str'>


Looking at network json
Attribute storage type is <class 'dict'>
Attribute type is <class 'str'>


Looking at network pkl
Attribute storage type is <class 'dict'>
Attribute type is <class 'str'>




array([[0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1490, 1490))

### Question 1c

In [56]:
# Fix Edgelist
nx.write_edgelist(G, "outputs/polblogs_fixed.edgelist", data=["website", "political_orientation"],  delimiter=',')
edge_list = nx.read_edgelist("outputs/polblogs_fixed.edgelist", create_using=nx.DiGraph,  delimiter=',')

print(f"Network: edgelist")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.edgelist') / 1024)} kilobytes")
print(f"Node Count: {edge_list.number_of_nodes()}")
print(f"Edge Count: {edge_list.number_of_edges()}")
print(f"Is directed: {nx.is_directed(edge_list)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(edge_list, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(edge_list, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, edge_list)}\n")

Network: edgelist
File size: 158 kilobytes
Node Count: 1224
Edge Count: 19025
Is directed: True
Website Attr Length: 0
Political Orientation Attr Length: 0
Graphs are equal: False



In [57]:
# Fix GraphML
nx.write_graphml(G, "outputs/polblogs_fixed.graphml")
GraphML_fixed = nx.read_graphml("outputs/polblogs_fixed.graphml")

print(f"Network: GraphML")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.graphml') / 1024)} kilobytes ")
print(f"Node Count: {GraphML_fixed.number_of_nodes()}")
print(f"Edge Count: {GraphML_fixed.number_of_edges()}")
print(f"Is directed: {nx.is_directed(GraphML_fixed)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(GraphML_fixed, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(GraphML_fixed, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, GraphML_fixed)}\n")

# look for the differences
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.edges == G.edges)
print(GraphML_fixed.edges == G.edges)
print(nx.get_node_attributes(GraphML_fixed, 'political_orientation')==nx.get_node_attributes(G, 'political_orientation'))
print(nx.get_node_attributes(GraphML_fixed, 'website')==nx.get_node_attributes(G, 'website'))

print("Looking at nodes and data....")
print(G.nodes(data=True) == GraphML_fixed.nodes(data=True))
#print(f"Same shape: {nx.is_isomorphic(G, GraphML_fixed)}")
print(type(G))
print(type(GraphML_fixed))


# Check IDs
node_ids = list(G)
graph_node_ids = list(GraphML_fixed)
print(graph_node_ids == node_ids)

# Everything checks out. Maybe some oddness with the python object itself. 
# Now check based on docstring
print("Looking at docstring evaluations...")
print(GraphML_fixed.adj == G.adj)
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.graph == G.graph)
print("Done.")

# The graphs are not equal, let's print both
print(f"GraphM:: {GraphML_fixed.graph}")
print(f"Original: {G.graph}")

# The original is just an empty list. So reset 
GraphML_fixed.graph = {}
print("Updated new graph. Equal?")
print(GraphML_fixed.adj == G.adj)
print(GraphML_fixed.nodes == G.nodes)
print(GraphML_fixed.graph == G.graph)
print(nx.utils.graphs_equal(GraphML_fixed, G))
# Fixed it!

Network: GraphML
File size: 912 kilobytes 
Node Count: 1490
Edge Count: 19025
Is directed: True
Website Attr Length: 1490
Political Orientation Attr Length: 1490
Graphs are equal: False

True
True
True
True
True
Looking at nodes and data....
True
<class 'networkx.classes.digraph.DiGraph'>
<class 'networkx.classes.digraph.DiGraph'>
True
Looking at docstring evaluations...
True
True
False
Done.
GraphM:: {'node_default': {}, 'edge_default': {}}
Original: {}
Updated new graph. Equal?
True
True
True
True


In [58]:
# dense adjacency matrix
nmp = nx.to_numpy_array(G)
with open("outputs/polblogs_fixed.npy", 'wb') as gfile:
    np.save(gfile, nmp)

# NP dense ajacency matrix
# Pickle binary serialization
with open("outputs/polblogs_fixed.npy", 'rb') as gfile:
    back_np = np.load(gfile)
    np_network = nx.from_numpy_array(back_np, create_using=nx.DiGraph)

print(f"\n\nNetwork: NP Array")
print(f"File size: {round(os.path.getsize(f'outputs/polblogs_fixed.edgelist') / 1024)} kilobytes")
print(f"Node Count: {np_network.number_of_nodes()}")
print(f"Edge Count: {np_network.number_of_edges()}")
print(f"Is directed: {nx.is_directed(np_network)}")
print(f"Website Attr Length: {len(nx.get_node_attributes(np_network, 'website'))}")
print(f"Political Orientation Attr Length: {len(nx.get_node_attributes(np_network, 'political_orientation'))}")
print(f"Graphs are equal: {nx.utils.graphs_equal(G, np_network)}\n")



Network: NP Array
File size: 158 kilobytes
Node Count: 1490
Edge Count: 19025
Is directed: True
Website Attr Length: 0
Political Orientation Attr Length: 0
Graphs are equal: False



### Question 1e

In [65]:
baby_graph = nx.path_graph(5)
print(baby_graph.nodes(data=True))


[(0, {}), (1, {}), (2, {}), (3, {}), (4, {})]


In [66]:
# Edgelist
nx.write_edgelist(G, "outputs/baby_graph.edgelist")
baby_edge = nx.read_edgelist("outputs/baby_graph.edgelist")
baby_edge.nodes(data=True)
# labels stored as strings

NodeDataView({'0': {}, '22': {}, '54': {}, '84': {}, '154': {}, '322': {}, '366': {}, '433': {}, '482': {}, '574': {}, '640': {}, '641': {}, '643': {}, '663': {}, '1244': {}, '1434': {}, '1': {}, '17': {}, '23': {}, '28': {}, '51': {}, '98': {}, '143': {}, '153': {}, '158': {}, '171': {}, '179': {}, '203': {}, '237': {}, '247': {}, '331': {}, '362': {}, '388': {}, '390': {}, '430': {}, '445': {}, '476': {}, '494': {}, '543': {}, '546': {}, '560': {}, '562': {}, '563': {}, '565': {}, '571': {}, '622': {}, '649': {}, '668': {}, '686': {}, '753': {}, '755': {}, '4': {}, '513': {}, '719': {}, '1436': {}, '5': {}, '736': {}, '7': {}, '39': {}, '240': {}, '263': {}, '266': {}, '277': {}, '386': {}, '392': {}, '511': {}, '566': {}, '573': {}, '590': {}, '595': {}, '603': {}, '633': {}, '8': {}, '144': {}, '209': {}, '740': {}, '995': {}, '9': {}, '71': {}, '300': {}, '460': {}, '471': {}, '534': {}, '1062': {}, '10': {}, '1054': {}, '11': {}, '95': {}, '12': {}, '149': {}, '296': {}, '439': {

In [74]:
# GraphML
nx.write_graphml(G, "outputs/baby_graph.graphml")
baby_graphml = nx.read_graphml("outputs/baby_graph.graphml")
baby_graphml.nodes(data=True)
# labels also stored as strings

NodeDataView({'0': {'website': '100monkeystyping.com', 'political_orientation': 'liberal'}, '1': {'website': '12thharmonic.com/wordpress', 'political_orientation': 'liberal'}, '2': {'website': '40ozblog.blogspot.com', 'political_orientation': 'liberal'}, '3': {'website': '4lina.tblog.com', 'political_orientation': 'liberal'}, '4': {'website': '750volts.blogspot.com', 'political_orientation': 'liberal'}, '5': {'website': '95theses.blogspot.com', 'political_orientation': 'liberal'}, '6': {'website': 'abbadabbaduo.blogspot.com', 'political_orientation': 'liberal'}, '7': {'website': 'aboutpolitics.blogspot.com', 'political_orientation': 'liberal'}, '8': {'website': 'achingandspinning.blogspot.com', 'political_orientation': 'liberal'}, '9': {'website': 'ackackack.com', 'political_orientation': 'liberal'}, '10': {'website': 'adamtalib.blogspot.com', 'political_orientation': 'liberal'}, '11': {'website': 'adviceforlefty.blogspot.com', 'political_orientation': 'liberal'}, '12': {'website': 'ag

In [75]:
# now try with int
baby_edge = nx.read_edgelist("outputs/baby_graph.edgelist", nodetype=int)
print(baby_edge.nodes(data=True))

baby_graphml = nx.read_graphml("outputs/baby_graph.graphml",  node_type=int)
print(baby_graphml.nodes(data=True))

# All fixed! 


[(0, {}), (22, {}), (54, {}), (84, {}), (154, {}), (322, {}), (366, {}), (433, {}), (482, {}), (574, {}), (640, {}), (641, {}), (643, {}), (663, {}), (1244, {}), (1434, {}), (1, {}), (17, {}), (23, {}), (28, {}), (51, {}), (98, {}), (143, {}), (153, {}), (158, {}), (171, {}), (179, {}), (203, {}), (237, {}), (247, {}), (331, {}), (362, {}), (388, {}), (390, {}), (430, {}), (445, {}), (476, {}), (494, {}), (543, {}), (546, {}), (560, {}), (562, {}), (563, {}), (565, {}), (571, {}), (622, {}), (649, {}), (668, {}), (686, {}), (753, {}), (755, {}), (4, {}), (513, {}), (719, {}), (1436, {}), (5, {}), (736, {}), (7, {}), (39, {}), (240, {}), (263, {}), (266, {}), (277, {}), (386, {}), (392, {}), (511, {}), (566, {}), (573, {}), (590, {}), (595, {}), (603, {}), (633, {}), (8, {}), (144, {}), (209, {}), (740, {}), (995, {}), (9, {}), (71, {}), (300, {}), (460, {}), (471, {}), (534, {}), (1062, {}), (10, {}), (1054, {}), (11, {}), (95, {}), (12, {}), (149, {}), (296, {}), (439, {}), (479, {}),

### Question 1f

In [ ]:
# First with the write edge method of storing 

# size of current network
node_count = len(G.nodes)
write_link_count = 16718 # (its different because no edgeless nodes)

# I anticipate that edgewrite scales at O(N) except where N equals links
# So what proportion of links to nodes?
potential_links = node_count ** 2 # 2220100
link_proportion = write_link_count / potential_links # 0.00856943380928787 or .86%
# so number of links on a 10**6 network would be
big_potential_links = (10 ** 6 )**2
# and then we want .68% of that so
big_link_count = big_potential_links * link_proportion
# and now get kb size in proportion 
# link_count / file size = big_link_count / file size)
new_size = (big_link_count * 214)/write_link_count
print(new_size)

96392054.41196343


In [90]:
# Now repeat with adjacency matrix 

# size of current network
node_count = len(G.nodes)

# I anticipate that adjacency network scales at O(N^2) because all links are represented
# So we don't need to deal with link count, just N
big_node_count = 10 ** 6

# so we have big_N^2/file_size = N^2/file_size
# so (BigN^2 * file_size) / N^2
new_size = (big_link_count ** 2 * 17345)/node_count ** 2

print(new_size)

4.430220429819246e+17
